# Tutorial to Least Squares and Recursive Least Squares applied to Geolocation simulation

## Background
### Least Squares application for Geolocation

Small math background how it's formulated
<font color ='red'> TODO </font>

#### Python imports
Prior to beginning this jupyter-notebook tutorial, we can include some python imports: 
<font color ='red'> TODO </font>

In [ ]:
import os
import numpy as np
import geolocation.estimator as estimator
from geolocation.estimator import EstimatorAoA
import geolocation.sensor as sensor
import geolocation.controller as controller

from geolocation.analysis import * 
from geolocation.noise import GaussianNoise

###  Small introduction to Least Squares:

#### Least Squares typical formulation 

<font color ='red'> TODO </font>

#### Least Squares for geolocation

<font color ='red'> TODO </font>


In [ ]:
class LeastSQ(EstimatorAoA):
    def __init__(self, name='leastSQ_default'):
        super().__init__(name)

    def estimate(self, emitter):
        # pos_est , _ , _ , _ = np.linalg.lstsq(self.A_mat, self.b_vec)
        pos_est = np.linalg.inv(self.A_mat.T @ self.A_mat) @ self.A_mat.T @ self.b_vec
        self.estimate_ans = pos_est
        self.estimate_error(emitter)

### Batch Least Squares

Why can't we use all the prior angle information?  
Yes we can!  
<font color ='red'> TODO </font>

#### Batch Least Squares Linear Algebra

<font color ='red'> TODO </font>

In [ ]:
class BatchedLeastSQ(EstimatorAoA):
    def __init__(self, name='leastSQ_default'):
        super().__init__(name)

    def estimate(self, emitter):
        # pos_est , _ , _ , _ = np.linalg.lstsq(self.A_mat, self.b_vec)
        pos_est = np.linalg.inv(self.A_mat.T @ self.A_mat) @ self.A_mat.T @ self.b_vec
        self.estimate_ans = pos_est
        self.estimate_error(emitter)

### Recursive Least Squares

Is there a way to make this "online" estimation instead on data that is streamed?  
Or  
Do we have to keep all the previous data for our estimate?  

<font color ='red'> TODO </font>

#### Recursive Least Squares Math

<font color ='red'> TODO </font>

In [ ]:
class RecursiveLeastSQ(EstimatorAoA):
    def __init__(self, lambda_, P0=1e-3, name='recursiveLeastSQ_default'):
        Q = ((1 - lambda_) / lambda_) * P0 * np.eye(3)
        R = 1.0
        super().__init__(Q=Q,R=R,P0=1e3,name=name)

    def generate_est_matrices(self, slist):
        super().generate_est_matrices(slist)
        self.R = self.R*np.eye(self.A_mat.shape[0])

    def estimate(self, emitter):
        # Prediction
        pass



#### Initialize your Estimator Here: 

In [ ]:
## INITIALIZE ESTIMATOR HERE
myEstimator = LeastSQ()

#### Geolocation simulator run: 

In [ ]:
## INITIALIZE ENVIRONMENT 
## INITIALIZE SENSOR AND EMITTER OBJECTS
mu, sig = 0, 17.2
myNoise = GaussianNoise(mu=mu, sigma=sig)

# Set sensors, on top of brain locations. 
a = [1,0,0.8]
b = [-1,0,0.8]
c = [0,1,0.5]
d = [0,-1, 0.5]

# Set z, the seizure inside temporal lobe bounds: 
# bounds = [[-0.6, 0.6],[-0.25, 0.5],[-0.4, 0.1]]
z = [0.2,-0.1,-0.2]

aS = sensor.Sensor("a",a[0],a[1],a[2])
bS = sensor.Sensor("b",b[0],b[1],b[2])
cS = sensor.Sensor("c",c[0],c[1],c[2])
dS = sensor.Sensor("d",d[0],d[1],d[2])

mySeizure = sensor.Sensor("Seizure",z[0],z[1],z[2])
sensor_list = [aS,bS,cS,dS]
num_sens = 4
sensor_list = sensor_list[:num_sens]

for sen in sensor_list:
    sen.set_noise(myNoise)

## INITIALIZE CONTROLLER 
rng = np.random.default_rng(10)
mc_count = 1000
## TODO, Controller should take in environment, and run environment setup here, rather than manually 
#        affecting the sensor relocation earlier in this main script 
myController = controller.Controller(mySeizure, 
                                    sensor_list, 
                                    myEstimator,
                                    mc_count = mc_count, rng=rng)
myController.run_simulation()

In [ ]:
# TODO: consider removing this from controller and have it all from the estimator object instead. 
# Controller only runs the simulation, not keeps values from it. 
all_pos_est = myController.all_pos_est
all_pos_err = myController.all_pos_err


# plotVer_2D(mySeizure,sensor_list,estimates=all_pos_est)
plot_err(mc_count, est=all_pos_est, err=all_pos_err, scale=9)

# KF Specific
#kf_params = myRLS.get_KF_update_metrics()
#p_hist = kf_params["P_hist"]

#plot_track_cov(mySeizure, p_hist=p_hist)
